In [ ]:
from pathlib import Path
import sys

project_root = next(
    (
        directory
        for directory in (Path.cwd(), *Path.cwd().parents)
        if (directory / "utils" / "utls_google_drive_auth.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from inside the project folder."
    )

sys.path.insert(0, str(project_root))

from utils.utls_work_images import list_images, load_image_as_array, build_dataset
from utils.utls_install_libs import install_missing_libs
from utils.utls_ia_models import build_fast_cnn

bibliotecas_necessarias = [
    'google-api-python-client',
    'google-auth-httplib2',
    'google-auth-oauthlib',
    'pillow',
    'numpy',
    'opencv-python',
    'tensorflow'
]
install_missing_libs(bibliotecas_necessarias)

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
images_expulsion = list_images("expulsion")

In [ ]:
primeiro_id = images_expulsion[0]['id']
matriz_imagem = load_image_as_array(primeiro_id, 224, 224)

In [ ]:
X_dados, y_dados = build_dataset(limit_per_class_collected=10,
                                 folder_name_collected="expulsion")
print(f"Formato final de X: {X_dados.shape}")
print(f"Formato final de y: {y_dados.shape}")

In [ ]:
X_exp, y_exp = build_dataset(limit_per_class_collected=50, folder_name_collected="expulsion")
X_inc, y_inc = build_dataset(limit_per_class_collected=50, folder_name_collected="incubation")

# Juntar os tensores num dataset único
X_completo = np.concatenate((X_exp, X_inc), axis=0)
y_completo = np.concatenate((y_exp, y_inc), axis=0)

# Dividir os dados em treino (80%) e teste (20%) baralhando-os
X_train, X_test, y_train, y_test = train_test_split(X_completo, y_completo, test_size=0.2, random_state=42)

print(f"Dataset de Treino formatado: {X_train.shape}")

In [ ]:
modelo_reator = build_fast_cnn()
historico = modelo_reator.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))